## Pre-Shutdown Matryoshka Hybrid Models

Recent group experiments revealed that the catastrophic drop in performance on the test set is driven entirely by the dark market shutdown at Time Step 43.

To properly evaluate our architectures without the interference of this extreme temporal anomaly, we isolate the dataset to the stable "pre-shutdown" regime:
* **Train:** Time Steps 1 - 32
* **Validation:** Time Steps 33 - 37
* **Test:** Time Steps 38 - 42

We will train two separate feature extractors:
1.  **Regular Matryoshka GCN:** Standard neighborhood aggregation.
2.  **Matryoshka Skip-GCN:** Includes a residual connection to prevent dilution of local features.

Both models will be trained on the 93 stable local features with Early Stopping. Their nested embeddings ($d \in \{4, 8, 16, 32, 64, 128\}$) will be extracted and fed into our `ExtraTrees` baseline alongside the full 165 tabular features.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from pathlib import Path  # Replaced google.colab drive
from torch_geometric.nn import GCNConv
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

# 1. Set Path & Load Data
# Pointing up one level to the Dataset folder, just like the EDA notebook
DATA_DIR = Path("../Dataset")
PROCESSED_PATH = DATA_DIR / "processed" / "transaction_graph_v1.pt"

print(f"Loading data from: {PROCESSED_PATH.resolve()}")

# PyTorch can read directly from the Path object
data_dict, _, _ = torch.load(PROCESSED_PATH, weights_only=False, map_location="cpu")

X = data_dict['x']
y = data_dict['y']
edge_index = data_dict['edge_index']
time_steps = X[:, 0].numpy()

# 2. NEW SPLIT (Pre-Shutdown)
train_mask = (time_steps >= 1) & (time_steps <= 32)
val_mask = (time_steps >= 33) & (time_steps <= 37)
test_mask = (time_steps >= 38) & (time_steps <= 42)

labeled_mask = (y == 0) | (y == 1)

train_idx = torch.tensor(np.where(train_mask & labeled_mask.numpy())[0], dtype=torch.long)
val_idx = torch.tensor(np.where(val_mask & labeled_mask.numpy())[0], dtype=torch.long)
test_idx = torch.tensor(np.where(test_mask & labeled_mask.numpy())[0], dtype=torch.long)

# 3. Isolate the 93 Local Features
X_local = X[:, :93]

# 4. Handle Class Imbalance Weights
num_licit = (y[train_idx] == 0).sum().item()
num_illicit = (y[train_idx] == 1).sum().item()
pos_weight = torch.tensor([num_licit / num_illicit], dtype=torch.float32)

print(f"Pre-Shutdown Split Ready.")
print(f"Train nodes: {len(train_idx)}, Val nodes: {len(val_idx)}, Test nodes: {len(test_idx)}")

Mounted at /content/drive
Pre-Shutdown Split Ready.
Train nodes: 28938, Val nodes: 4503, Test nodes: 6436


In [4]:
class MatryoshkaGCN(torch.nn.Module):
    """Standard GCN with Matryoshka nested output heads."""
    def __init__(self, in_channels, max_dim=128, nested_dims=[4, 8, 16, 32, 64, 128]):
        super(MatryoshkaGCN, self).__init__()
        self.nested_dims = nested_dims

        self.conv1 = GCNConv(in_channels, max_dim * 2)
        self.conv2 = GCNConv(max_dim * 2, max_dim)

        self.classifiers = torch.nn.ModuleDict({
            str(dim): torch.nn.Linear(dim, 1) for dim in nested_dims
        })

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.5, training=self.training)
        embeddings = self.conv2(h, edge_index)

        outputs = {}
        for dim in self.nested_dims:
            outputs[str(dim)] = self.classifiers[str(dim)](embeddings[:, :dim])
        return embeddings, outputs


class MatryoshkaSkipGCN(torch.nn.Module):
    """GCN with a residual connection from the input features."""
    def __init__(self, in_channels, max_dim=128, nested_dims=[4, 8, 16, 32, 64, 128]):
        super(MatryoshkaSkipGCN, self).__init__()
        self.nested_dims = nested_dims

        self.conv1 = GCNConv(in_channels, max_dim * 2)
        self.conv2 = GCNConv(max_dim * 2, max_dim)
        self.skip = torch.nn.Linear(in_channels, max_dim)

        self.classifiers = torch.nn.ModuleDict({
            str(dim): torch.nn.Linear(dim, 1) for dim in nested_dims
        })

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.5, training=self.training)

        # Add the skip connection to the final embeddings
        embeddings = self.conv2(h, edge_index) + self.skip(x)

        outputs = {}
        for dim in self.nested_dims:
            outputs[str(dim)] = self.classifiers[str(dim)](embeddings[:, :dim])
        return embeddings, outputs

In [7]:
def train_and_evaluate_hybrid(model, model_name="GNN"):
    y_float = y.float().unsqueeze(1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def calculate_loss(mask_idx):
        _, outputs = model(X_local, edge_index)
        total_loss = 0
        for dim in model.nested_dims:
            pred = outputs[str(dim)][mask_idx]
            total_loss += criterion(pred, y_float[mask_idx])
        return total_loss

    # 1. Train with Early Stopping
    patience = 40
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    print(f"\n--- Training {model_name} ---")
    for epoch in range(1, 301):
        model.train()
        optimizer.zero_grad()
        train_loss = calculate_loss(train_idx)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = calculate_loss(val_idx).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        # ADDED PRINT STATEMENT HERE
        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss:.4f}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}!")
            break

    model.load_state_dict(best_model_state)
    print("Best model weights restored.")

    # 2. Extract Embeddings and Train ExtraTrees
    model.eval()
    with torch.no_grad():
        full_embeddings, _ = model(X_local, edge_index)

    X_full_np = X.numpy()
    y_train = y[train_idx].numpy()
    y_test = y[test_idx].numpy()

    results = []
    print(f"\n--- Evaluating Hybrid ExtraTrees ({model_name}) ---")

    for dim in model.nested_dims:
        mrl_slice = full_embeddings[:, :dim].numpy()
        X_hybrid = np.hstack([X_full_np, mrl_slice])

        et = ExtraTreesClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        et.fit(X_hybrid[train_idx], y_train)

        y_prob = et.predict_proba(X_hybrid[test_idx])[:, 1]
        pr_auc = average_precision_score(y_test, y_prob)

        results.append({"MRL_Dim": dim, "Test_PRAUC": pr_auc})
        print(f"MRL {dim:3d} -> Test PR-AUC: {pr_auc:.4f}")

    return pd.DataFrame(results).set_index("MRL_Dim")

In [8]:
# Test 1: Regular Matryoshka
model_regular = MatryoshkaGCN(in_channels=X_local.shape[1])
results_regular = train_and_evaluate_hybrid(model_regular, "Regular Matryoshka GCN")

# Test 2: Skip-Connection Matryoshka
model_skip = MatryoshkaSkipGCN(in_channels=X_local.shape[1])
results_skip = train_and_evaluate_hybrid(model_skip, "Matryoshka Skip-GCN")


--- Training Regular Matryoshka GCN ---
Epoch 010 | Train Loss: 5.9234 | Val Loss: 6.1482
Epoch 020 | Train Loss: 5.1576 | Val Loss: 6.1713
Epoch 030 | Train Loss: 4.6963 | Val Loss: 6.0506
Epoch 040 | Train Loss: 4.3988 | Val Loss: 5.8844
Epoch 050 | Train Loss: 4.1987 | Val Loss: 5.8466
Epoch 060 | Train Loss: 3.9978 | Val Loss: 5.7804
Epoch 070 | Train Loss: 3.8814 | Val Loss: 5.6724
Epoch 080 | Train Loss: 3.7865 | Val Loss: 5.5904
Epoch 090 | Train Loss: 3.6489 | Val Loss: 5.4042
Epoch 100 | Train Loss: 3.5693 | Val Loss: 5.2784
Epoch 110 | Train Loss: 3.4462 | Val Loss: 5.1003
Epoch 120 | Train Loss: 3.3791 | Val Loss: 4.8962
Epoch 130 | Train Loss: 3.3214 | Val Loss: 4.6981
Epoch 140 | Train Loss: 3.2250 | Val Loss: 4.5324
Epoch 150 | Train Loss: 3.1499 | Val Loss: 4.3913
Epoch 160 | Train Loss: 3.0710 | Val Loss: 4.2770
Epoch 170 | Train Loss: 3.0191 | Val Loss: 4.1174
Epoch 180 | Train Loss: 2.9845 | Val Loss: 4.0458
Epoch 190 | Train Loss: 2.9233 | Val Loss: 3.9293
Epoch 200

In [9]:
import os
import torch

# 1. Define and create a folder for your saved models
SAVE_DIR = f"{PROJECT_DIR}/Models"
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Save the Regular Matryoshka weights
regular_path = f"{SAVE_DIR}/matryoshka_regular_pre_shutdown.pt"
torch.save(model_regular.state_dict(), regular_path)
print(f"Saved Regular Matryoshka to: {regular_path}")

# 3. Save the Skip-GCN Matryoshka weights
skip_path = f"{SAVE_DIR}/matryoshka_skip_pre_shutdown.pt"
torch.save(model_skip.state_dict(), skip_path)
print(f"Saved Skip-GCN Matryoshka to: {skip_path}")

Saved Regular Matryoshka to: /content/drive/MyDrive/2026 Deep Learning Bitcoin Fraud Detection/Models/matryoshka_regular_pre_shutdown.pt
Saved Skip-GCN Matryoshka to: /content/drive/MyDrive/2026 Deep Learning Bitcoin Fraud Detection/Models/matryoshka_skip_pre_shutdown.pt
